# Lesson 06 · The seven stops, in code

This notebook replays the lecture inside a real model, **GPT-2 small (124M parameters)**, loaded from Hugging Face.
No training happens here. We only *read* the model and *run* it, one stop at a time.

| Stop | What you will see with your own eyes |
|---|---|
| 1 · tokens | the real 50,257-piece vocabulary, and why *strawberry* is hard |
| 2 · embeddings | the 50,257 × 768 table, and the row of `" berry"` |
| 3 · positions | the 1024 × 768 position table, and why `car @ 0 ≠ car @ 2` |
| 4 · attention | one head built by hand in ~10 lines, matched against the library, with and without the mask |
| 5 · the block | norm, attention, residual, norm, FFN, residual, read straight from the code |
| 6 · sampling | the 50,257 logits, weight tying, temperature, and the generation loop |
| 7 · the model | the 124,439,808 parameters, counted by hand |

Every section ends with a **Think about it** question. Answer it in a sentence before moving on.
Runs on the free Colab tier, CPU is enough. Expect ~1 minute to download the model the first time.

In [ ]:
# Setup (run once). If you are on Colab, this installs what is missing.
!pip -q install "transformers>=4.45" torch matplotlib numpy

import torch, numpy as np, matplotlib.pyplot as plt, time, math
from transformers import GPT2TokenizerFast, GPT2Tokenizer, GPT2LMHeadModel
torch.manual_seed(0)

tok   = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2", attn_implementation="eager")   # eager = lets us read attention maps
model.eval()
print("loaded GPT-2 small ·", sum(p.numel() for p in model.parameters()), "parameters")   # expect 124,439,808

In [ ]:
# Small plotting helper used throughout: a labelled heatmap for attention tables.
def heatmap(matrix, xlabels, ylabels, title="", ax=None, annotate=True, vmax=None):
    ax = ax or plt.gca()
    im = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=vmax if vmax is not None else max(1e-9, float(np.max(matrix))))
    ax.set_xticks(range(len(xlabels))); ax.set_xticklabels(xlabels, rotation=45, ha="right")
    ax.set_yticks(range(len(ylabels))); ax.set_yticklabels(ylabels)
    ax.set_title(title, fontsize=10)
    if annotate and matrix.shape[0] <= 12:
        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                v = matrix[i, j]
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if v > 0.5 * (vmax or matrix.max()) else "black")
    return im

def clean(tokens):
    # GPT-2 shows a leading space as "Ġ" and a newline as "Ċ". Make them readable.
    return [t.replace("Ġ", "␣").replace("Ċ", "⏎") for t in tokens]

## Stop 1 · Tokens: text becomes pieces

The model never sees text. It sees a sequence of **ids**, indices into a fixed vocabulary of 50,257 pieces.
The pieces were not chosen by hand. **Byte-Pair Encoding (BPE)** discovered them by counting.

In [ ]:
for text in ["strawberry", "a red car is fast", "O carro vermelho é rápido"]:
    ids = tok.encode(text)
    print(f"{text!r}")
    print("   ids    :", ids)
    print("   pieces :", clean(tok.convert_ids_to_tokens(ids)))
    print()
# Notice: "␣" marks a piece that starts with a space. " car" and "car" are DIFFERENT pieces.

**Where does 50,257 come from?** 256 byte symbols + 50,000 learned merges + 1 special token. Let us check it on the real tokenizer.

In [ ]:
import json
spec   = json.loads(tok.backend_tokenizer.to_str())      # the tokenizer's own recipe, as JSON
merges = spec["model"]["merges"]                          # the learned merges, in the order BPE discovered them
n_merges  = len(merges)
n_bytes   = sum(1 for piece in tok.get_vocab() if len(piece) == 1)   # the 256 byte symbols are the one-character pieces
n_special = len(tok.all_special_tokens)                   # <|endoftext|>
print(f"{n_bytes} bytes + {n_merges} merges + {n_special} special = {n_bytes + n_merges + n_special}")
print("vocabulary size reported by the tokenizer:", len(tok))

# The first merges BPE discovered on its corpus, the most frequent adjacent pairs:
first = [" + ".join(m) if isinstance(m, (list, tuple)) else m.replace(" ", " + ") for m in merges[:12]]
print("first merges:", first)

**Why not words? Why not letters?** Byte-level BPE means *nothing* is ever unknown. But the price depends on the language.

In [ ]:
en = ("The transformer processes every token in parallel. Attention lets each word look at the "
      "others and decide what it means in context.")
pt = ("O transformer processa todos os tokens em paralelo. A atenção permite que cada palavra olhe "
      "para as outras e decida o que significa no contexto.")
emoji = "🚗🚗🚗"

for name, text in [("English", en), ("Portuguese", pt), ("emoji", emoji)]:
    n_tok, n_words = len(tok.encode(text)), len(text.split())
    print(f"{name:10s} {n_tok:3d} tokens for {n_words:2d} words  →  {n_tok / max(1, n_words):.2f} tokens/word")

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(["English", "Portuguese"], [len(tok.encode(en)) / len(en.split()), len(tok.encode(pt)) / len(pt.split())],
       color=["#1E3261", "#6E9277"])
ax.set_ylabel("tokens per word"); ax.set_title("the same idea costs more in Portuguese")
plt.show()

> **Think about it.** The corpus that trained this BPE was mostly English. What does that imply for the API bill and for the context window of a Portuguese application?

## Stop 2 · Embeddings: the id becomes a vector

Turning an id into a vector is a **lookup**, no computation at all. The table `wte` has one row per piece and 768 columns.
The table is *learned* together with the rest of the model (it starts as random noise).

In [ ]:
wte = model.transformer.wte.weight.detach()          # the embedding table
print("table shape:", tuple(wte.shape))              # expect (50257, 768)

ids = tok.encode(" berry")
print('" berry" is', len(ids), "piece(s), id(s) =", ids)
row = wte[ids[0]]
print("first 8 of its 768 numbers:", np.round(row[:8].numpy(), 3))
print('"berry" (no space) has id(s)', tok.encode("berry"), "→ a different row")

fig, ax = plt.subplots(figsize=(10, 1.2))
ax.imshow(row.numpy()[None, :], cmap="RdBu", aspect="auto"); ax.set_yticks([]); ax.set_xlabel("768 dimensions")
ax.set_title('the row of " berry", as a ribbon of 768 numbers'); plt.show()

**No single dimension means anything.** Meaning is the *direction* of the whole vector. Similar words point in similar directions.

In [ ]:
def emb(word):
    ids = tok.encode(word)
    assert len(ids) == 1, f"{word!r} is not a single piece: {ids}"
    return wte[ids[0]]

def cosine(a, b): return float(torch.dot(a, b) / (a.norm() * b.norm()))

for w in [" truck", " vehicle", " red", " banana", " Tuesday"]:
    print(f'cos(" car", "{w}") = {cosine(emb(" car"), emb(w)):.3f}')

# A 2-D shadow of a few words (PCA by SVD). Distances are only suggestive, the real geometry lives in 768-D.
words = [" car", " truck", " bus", " bicycle", " red", " blue", " green", " king", " queen", " man", " woman", " banana", " apple"]
M = torch.stack([emb(w) for w in words]); M = M - M.mean(0)
U, S, Vt = torch.linalg.svd(M, full_matrices=False)
P = (M @ Vt[:2].T).numpy()
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(P[:, 0], P[:, 1], color="#1E3261")
for (x, y), w in zip(P, words): ax.annotate(w.strip(), (x, y), fontsize=9, xytext=(3, 3), textcoords="offset points")
ax.set_title("13 embeddings, projected to 2-D"); plt.show()

> **Think about it.** This table alone is 50,257 × 768 ≈ 38.6M numbers, one third of GPT-2. It is counted as *parameters*. Why? (Hint: who wrote the numbers in it?)

## Stop 3 · Positions: attention has no order

Attention scores tokens as a **set**, so `a red car` and `car red a` would be the same. The fix is a second table, `wpe`,
with one *learned* vector per position, **added** element-wise to the token vector.

In [ ]:
wpe = model.transformer.wpe.weight.detach()
print("position table shape:", tuple(wpe.shape))    # expect (1024, 768) → this 1024 is GPT-2's context window

fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(wpe[:256].numpy(), cmap="RdBu", aspect="auto", vmin=-0.3, vmax=0.3)
ax.set_xlabel("768 dimensions"); ax.set_ylabel("position (first 256 shown)")
ax.set_title("the learned position table has visible structure"); plt.show()

In [ ]:
car = emb(" car")
car_at_0, car_at_2 = car + wpe[0], car + wpe[2]
print("first 5 numbers of ' car'      :", np.round(car[:5].numpy(), 3))
print("first 5 numbers of pos 2       :", np.round(wpe[2][:5].numpy(), 3))
print("first 5 numbers of ' car' @ 2  :", np.round(car_at_2[:5].numpy(), 3), "  ← element-wise sum")
print(f"cos(car@0, car@2) = {cosine(car_at_0, car_at_2):.3f}  → same word, different vector")

> **Think about it.** The table has 1024 rows. What happens to the 1025th token? (This is where the context window limit physically lives.)

## Stop 4 · Attention: a word decides what it means by looking around

We now build **one attention head by hand**, exactly the six steps of the lecture, on the sentence `a red car`,
and check that our numbers match the library's numbers. We use layer 0, head 0.

GPT-2 applies a LayerNorm before attention (pre-norm), so the vector that enters the head is `ln_1(embedding + position)`.
The three matrices `Wq, Wk, Wv` are stored side by side in `c_attn` (768 → 3 × 768) and then sliced into 12 heads of 64.

In [ ]:
sentence = "a red car"
ids = torch.tensor([tok.encode(sentence)])
labels = clean(tok.convert_ids_to_tokens(ids[0]))
print("pieces:", labels)                       # expect 3 pieces

LAYER, HEAD, D_HEAD = 0, 0, 64
block = model.transformer.h[LAYER]

with torch.no_grad():
    # the sum of stop 3, then the pre-attention norm
    x = model.transformer.wte(ids) + model.transformer.wpe(torch.arange(ids.shape[1]))
    x = block.ln_1(x)[0]                                              # (T, 768)

    # step 1 · q, k, v for every token (same three matrices for all tokens)
    W, b = block.attn.c_attn.weight, block.attn.c_attn.bias           # (768, 2304), (2304,)
    qkv = x @ W + b
    Q, K, V = qkv.split(768, dim=-1)
    sl = slice(HEAD * D_HEAD, (HEAD + 1) * D_HEAD)                    # this head's 64-dim slice
    q, k, v = Q[:, sl], K[:, sl], V[:, sl]
    print("q, k, v shapes:", tuple(q.shape), tuple(k.shape), tuple(v.shape))   # (3, 64) each

    # step 2 · scores, one number per pair (the token scores itself too)
    scores = q @ k.T
    # step 3 · scale by sqrt(d_k) = 8
    scores = scores / math.sqrt(D_HEAD)
    # the mask · future → -inf, so the softmax gives it weight 0
    T = scores.shape[0]
    causal = torch.tril(torch.ones(T, T, dtype=torch.bool))
    masked = scores.masked_fill(~causal, float("-inf"))
    # step 4 · softmax, rows become weights that sum to 1
    weights_full   = torch.softmax(scores, dim=-1)                    # what the table would be WITHOUT the mask
    weights_masked = torch.softmax(masked, dim=-1)                    # what GPT-2 actually uses
    # steps 5 and 6 · weight the values and sum
    z = weights_masked @ v                                            # (3, 64), one new vector per token

print("z shape:", tuple(z.shape))
print("rows sum to 1:", weights_masked.sum(-1).numpy().round(4))

In [ ]:
# Does our hand-made head match the library?
with torch.no_grad():
    out = model(ids, output_attentions=True)
lib = out.attentions[LAYER][0, HEAD]                                  # (T, T) attention weights, layer 0 head 0
print("matches the library:", torch.allclose(lib, weights_masked, atol=1e-4))

fig, axes = plt.subplots(1, 2, figsize=(8, 3.6))
heatmap(weights_full.numpy(),   labels, labels, "without the mask (BERT's world)", axes[0], vmax=1)
heatmap(weights_masked.numpy(), labels, labels, "with the mask (what GPT-2 uses)", axes[1], vmax=1)
plt.tight_layout(); plt.show()
# Read the last row: " car" sees everything either way, it was the last word. The mask only censors the rows above.

**Multi-head.** Twelve heads run in parallel, each with its own `Wq, Wk, Wv` on its own 64-dim slice. Their outputs are
concatenated (12 × 64 = 768) and mixed by one more learned matrix, `W⁰` (`c_proj`). Let us look at what the twelve heads
of one layer do on a sentence with an ambiguous pronoun, the example from *The Illustrated Transformer*.

In [ ]:
print("W⁰ (c_proj) shape:", tuple(block.attn.c_proj.weight.shape))   # (768, 768): concat of 12×64 in, 768 out

sentence = "The animal didn't cross the street because it was too tired"
ids = torch.tensor([tok.encode(sentence)])
labels = clean(tok.convert_ids_to_tokens(ids[0]))
with torch.no_grad():
    out = model(ids, output_attentions=True)

LAYER = 5                                         # try 3, 5, 8 and see how heads change with depth
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for h, ax in enumerate(axes.flat):
    heatmap(out.attentions[LAYER][0, h].numpy(), labels, labels, f"layer {LAYER} · head {h}", ax, annotate=False, vmax=1)
plt.suptitle("twelve ways of looking, at once · rows ask, columns answer", y=1.0); plt.tight_layout(); plt.show()

In [ ]:
# Where does " it" look, in each head? Rank the heads by the weight they put on " animal" vs " street".
it_pos     = labels.index("␣it")
animal_pos = labels.index("␣animal")
street_pos = labels.index("␣street")
rows = []
for L in range(12):
    for h in range(12):
        a = out.attentions[L][0, h]
        rows.append((L, h, float(a[it_pos, animal_pos]), float(a[it_pos, street_pos])))
rows.sort(key=lambda r: -r[2])
print("heads where ' it' attends most to ' animal':")
for L, h, wa, ws in rows[:6]:
    print(f"  layer {L:2d} head {h:2d}   animal={wa:.2f}   street={ws:.2f}")

> **Think about it.** Nobody told these heads to track pronouns. Pick the top head above and plot it. Then change the sentence to
> `"...because it was too wide"` (now *it* is the street). Does the same head switch? Roles emerge, and they are not always clean.

## Stop 5 · The block: attention, then a private think

GPT-2 stacks 12 identical blocks. Read one straight from the code: norm → attention → residual → norm → FFN (768 → 3072 → 768) → residual.

In [ ]:
print(model.transformer.h[0])

In [ ]:
def count(module): return sum(p.numel() for p in module.parameters())
blk = model.transformer.h[0]
parts = {"attention (Wq,Wk,Wv,W⁰)": count(blk.attn), "FFN (768→3072→768)": count(blk.mlp), "norms": count(blk.ln_1) + count(blk.ln_2)}
for k, v in parts.items(): print(f"{k:28s} {v:>10,}")
print(f"{'one block':28s} {count(blk):>10,}")        # expect 7,087,872
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(parts.keys(), parts.values(), color=["#1E3261", "#6E9277", "#B89A6E"]); ax.set_ylabel("parameters")
ax.set_title("inside one block, the FFN holds ~2/3"); plt.xticks(rotation=15); plt.show()

**The residual is a plain sum, zero parameters.** Let us run one block by hand and confirm the rail.

In [ ]:
ids = torch.tensor([tok.encode("a red car is fast")])
with torch.no_grad():
    x = model.transformer.wte(ids) + model.transformer.wpe(torch.arange(ids.shape[1]))
    a = blk.attn(blk.ln_1(x)); a = a[0] if isinstance(a, tuple) else a
    h = x + a                                 # attention, then ADD the input back (residual 1)
    y = h + blk.mlp(blk.ln_2(h))              # FFN, then ADD again (residual 2)
    ref = blk(x)
    ref = ref[0] if isinstance(ref, tuple) else ref
print("hand-made block matches the library:", torch.allclose(y, ref, atol=1e-4))
print("residual parameters:", 0, "· norm parameters:", count(blk.ln_1) + count(blk.ln_2), "(2×768 each, gain and shift)")

> **Think about it.** Attention is the only place where tokens talk to each other. Everything in the FFN happens to one token at a time. Why is that a good division of labour for parallel hardware?

## Stop 6 · Sampling: the output is a distribution, not an answer

After block 12 comes a final norm and a **readout** to 50,257 scores. The surprise: the readout matrix is the embedding table
of stop 2, transposed (**weight tying**). Softmax turns the scores into a distribution. Then a die is rolled.

In [ ]:
same_memory = model.lm_head.weight.data_ptr() == model.transformer.wte.weight.data_ptr()
print("lm_head IS the embedding table (weight tying):", same_memory)

prompt = "a red car is"
ids = torch.tensor([tok.encode(prompt)])
with torch.no_grad():
    logits = model(ids).logits[0, -1]               # one score per vocabulary piece, for the NEXT position
print("logits shape:", tuple(logits.shape))          # (50257,)
probs = torch.softmax(logits, dim=-1)
print("sum of the distribution:", f"{probs.sum():.4f}")

top = torch.topk(probs, 8)
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh([clean([t])[0] for t in tok.convert_ids_to_tokens(top.indices)][::-1], top.values.numpy()[::-1], color="#1E3261")
ax.set_title(f'next piece after "{prompt}"'); ax.set_xlabel("probability"); plt.show()
print(f"the other {len(probs) - 8:,} pieces share {1 - top.values.sum():.3f} of the probability")

**Temperature** reshapes the die before it is rolled. Divide the logits by T, then softmax.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, T in zip(axes, [0.2, 0.7, 1.5]):
    p = torch.softmax(logits / T, dim=-1)
    top = torch.topk(p, 8)
    ax.barh([clean([t])[0] for t in tok.convert_ids_to_tokens(top.indices)][::-1], top.values.numpy()[::-1], color="#6E9277")
    ax.set_title(f"T = {T} · top-8 hold {top.values.sum():.2f}"); ax.set_xlim(0, 1)
plt.tight_layout(); plt.show()

**The loop.** Generation is: run the model, sample one piece, append it, run again. Written by hand first, then with the library.

In [ ]:
def generate_by_hand(prompt, n_new=15, T=0.7):
    ids = tok.encode(prompt)
    for _ in range(n_new):
        with torch.no_grad():
            logits = model(torch.tensor([ids])).logits[0, -1]        # one forward pass per generated piece
        p = torch.softmax(logits / T, dim=-1)
        nxt = int(torch.multinomial(p, 1))                          # roll the die
        ids.append(nxt)                                             # append, and go again
    return tok.decode(ids)

for T in [0.2, 0.7, 1.5]:
    print(f"T={T}:", generate_by_hand("a red car is", T=T))

print()
lib = model.generate(torch.tensor([tok.encode("a red car is")]), max_new_tokens=15, do_sample=True, temperature=0.7,
                     pad_token_id=tok.eos_token_id)
print("library:", tok.decode(lib[0]))

> **Think about it.** Every new piece re-runs the model over everything so far. Why does an output token cost more than an input token on an API bill? And what does "reasoning" mode change in this loop?

## Stop 7 · The whole model, counted by hand

One embedding table, one position table, twelve identical blocks, one final norm, and a readout that reuses the embedding table.

In [ ]:
tf = model.transformer
pieces = {
    "embeddings  50,257×768": count(tf.wte),
    "positions   1,024×768":  count(tf.wpe),
    "12 blocks × 7,087,872":  sum(count(b) for b in tf.h),
    "final norm":             count(tf.ln_f),
}
for k, v in pieces.items(): print(f"{k:26s} {v:>12,}")
total = sum(pieces.values())
print(f"{'total':26s} {total:>12,}")
print("model.parameters() says  ", f"{sum(p.numel() for p in model.parameters()):,}")
print("readout (lm_head) adds   0  → it is the embedding table, tied")

fig, ax = plt.subplots(figsize=(8, 1.6))
left = 0
for (k, v), c in zip(pieces.items(), ["#1E3261", "#B89A6E", "#6E9277", "#999999"]):
    ax.barh([0], [v], left=left, color=c, label=k.split("  ")[0]); left += v
ax.set_yticks([]); ax.set_xlabel("parameters"); ax.legend(ncol=4, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.5))
ax.set_title(f"where the {total:,} live"); plt.show()

| model | blocks | width d | heads | context | parameters |
|---|---|---|---|---|---|
| GPT-2 small (2019) | 12 | 768 | 12 | 1,024 | 124M |
| GPT-2 XL (2019) | 48 | 1,600 | 25 | 1,024 | 1.5B |
| GPT-3 (2020) | 96 | 12,288 | 96 | 2,048 | 175B |

Same drawing. More blocks, wider d, longer context.

> **Think about it.** The FFN is ~2/3 of every block. Mixture-of-Experts models replace it with several experts and a router. Which stop does that change, and which stops stay exactly as you saw them here?

## Back to strawberry

The model never saw a single letter. It saw ids. Counting the r's is not a task it ever practised on the representation it has.

In [ ]:
word = "strawberry"
ids = tok.encode(word)
print("what you typed :", word, "→", word.count("r"), "r's")
print("what it saw    :", ids, "→", clean(tok.convert_ids_to_tokens(ids)))
print("letters visible to the model: none. Each id is a row of 768 numbers, not a string.")

> **Think about it.** Which stop explains *strawberry*? Which stop explains a confidently invented date? Which stop explains why the same question gets different answers?

That is the whole drawing. From here on, we use it. Next: the exercise notebook, where you compare GPT-2 with two modern models for a real decision.